# Project 5: E-commerce Analytics
## Customer Analytics & Recommendation Insights

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
from utils.data_analysis_utils import utils
print("✅ Imports OK")


In [ ]:
np.random.seed(42)
n_cust, n_prod, n_tx = 5000, 100, 50000

# customers
seg_map = {'Premium':(500,200,15,0.05),'Regular':(200,80,8,0.15),
           'Occasional':(80,40,3,0.30),'New':(50,30,1,0.40)}
cust_rows = []
for i in range(n_cust):
    seg = np.random.choice(list(seg_map.keys()), p=[0.15,0.35,0.35,0.15])
    mu,sd,freq,churn = seg_map[seg]
    cust_rows.append({
        'CustomerID':f'C{str(i+1).zfill(5)}','Segment':seg,
        'Age':np.random.randint(18,70),'Gender':np.random.choice(['M','F']),
        'Location':np.random.choice(['Urban','Suburban','Rural'],p=[0.5,0.35,0.15]),
        'Avg_Spend':max(5,np.random.normal(mu,sd)),
        'Loyalty':np.random.choice(['Bronze','Silver','Gold','Platinum'],p=[0.5,0.3,0.15,0.05]),
    })
custs = pd.DataFrame(cust_rows)

# products
cats = ['Electronics','Clothing','Books','Home','Sports','Beauty','Toys']
prods = pd.DataFrame({
    'ProductID':[f'P{str(i+1).zfill(4)}' for i in range(n_prod)],
    'Category': np.random.choice(cats, n_prod),
    'Price':    np.random.uniform(10,500,n_prod).round(2),
    'Rating':   np.random.uniform(3,5,n_prod).round(1),
})

# transactions
disc_map={'Bronze':0,'Silver':0.05,'Gold':0.10,'Platinum':0.15}
tx_rows=[]
date_range=pd.date_range('2023-07-01','2024-12-31',freq='h')
for i in range(n_tx):
    c = custs.sample(1).iloc[0]
    p = prods.sample(1).iloc[0]
    q = np.random.choice([1,2,3,4],p=[0.6,0.25,0.1,0.05])
    d = disc_map[c['Loyalty']]
    tx_rows.append({
        'TxID':f'T{str(i+1).zfill(8)}',
        'CustomerID':c['CustomerID'],'ProductID':p['ProductID'],
        'Category':p['Category'],'Quantity':q,'UnitPrice':p['Price'],
        'Discount':d,'TxDate':np.random.choice(date_range),
        'TotalAmount':round(q*p['Price']*(1-d),2),
        'PaymentMethod':np.random.choice(['Credit Card','PayPal','Debit Card','Apple Pay'],
                                          p=[0.4,0.3,0.2,0.1]),
    })
txs = pd.DataFrame(tx_rows)
print(f"✅ {len(custs):,} customers | {len(txs):,} transactions | Revenue ${txs['TotalAmount'].sum():,.2f}")


In [ ]:
os.makedirs('visualizations', exist_ok=True)

# CLV by segment
clv = txs.groupby('CustomerID').agg(
    Total_Spend=('TotalAmount','sum'),
    Orders=('TxID','count'),
    Avg_Order=('TotalAmount','mean')
).merge(custs[['CustomerID','Segment','Loyalty']], on='CustomerID')

seg_clv = clv.groupby('Segment')[['Total_Spend','Orders','Avg_Order']].mean().round(2)
fig, axes = plt.subplots(1,2,figsize=(14,6))
seg_clv.sort_values('Total_Spend').plot.barh(y='Total_Spend',ax=axes[0],legend=False)
axes[0].set_title('Avg CLV by Segment', fontweight='bold')
seg_clv.sort_values('Orders').plot.bar(y='Orders',ax=axes[1],legend=False,color='teal')
axes[1].set_title('Avg Orders by Segment', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/customer_clv.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: customer_clv.png")
print(seg_clv)


In [ ]:
# Category revenue
cat_rev = txs.groupby('Category')['TotalAmount'].sum().sort_values(ascending=False)
fig, axes = plt.subplots(1,2,figsize=(14,6))
axes[0].pie(cat_rev.values, labels=cat_rev.index, autopct='%1.1f%%')
axes[0].set_title('Revenue by Category', fontweight='bold')
cat_rev.plot.barh(ax=axes[1], color='coral')
axes[1].set_title('Revenue by Category (Bar)', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/category_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: category_performance.png")


In [ ]:
# K-Means customer clustering
X_cl = clv[['Total_Spend','Orders','Avg_Order']].fillna(0)
sc   = StandardScaler()
X_sc = sc.fit_transform(X_cl)

inertias = [KMeans(n_clusters=k,random_state=42,n_init=10).fit(X_sc).inertia_ for k in range(2,9)]
km = KMeans(n_clusters=4,random_state=42,n_init=10)
clv['Cluster'] = km.fit_predict(X_sc)

fig, axes = plt.subplots(1,2,figsize=(14,6))
axes[0].plot(range(2,9),inertias,marker='o')
axes[0].axvline(4,color='red',linestyle='--',label='K=4')
axes[0].set_title('Elbow Method', fontweight='bold'); axes[0].legend()

cluster_means = clv.groupby('Cluster')['Total_Spend'].mean()
axes[1].bar(cluster_means.index, cluster_means.values/1000, color='purple')
axes[1].set_title('Avg Spend by Cluster', fontweight='bold')
axes[1].set_ylabel('Avg Spend ($K)')
plt.tight_layout()
plt.savefig('visualizations/customer_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: customer_clusters.png")


In [ ]:
# Churn risk
last_tx   = txs.groupby('CustomerID')['TxDate'].max()
max_date  = txs['TxDate'].max()
recency   = (max_date - last_tx).dt.days

clv['Recency'] = clv['CustomerID'].map(recency).fillna(999)
def churn_risk(row):
    thresholds = {'Premium':45,'Regular':60,'Occasional':90,'New':30}
    t = thresholds.get(row['Segment'],60)
    return 'High' if row['Recency']>t else ('Medium' if row['Recency']>t//2 else 'Low')
clv['Churn_Risk'] = clv.apply(churn_risk, axis=1)

risk_counts = clv['Churn_Risk'].value_counts()
fig, ax = plt.subplots(figsize=(8,6))
ax.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%',
       colors=['green','orange','red'])
ax.set_title('Customer Churn Risk', fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/churn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

utils.plot_correlation_matrix(clv[['Total_Spend','Orders','Avg_Order','Recency']],
                               save_path='visualizations/correlation_matrix.png')
custs.to_csv('customer_data.csv', index=False)
txs.to_csv('transaction_data.csv', index=False)
clv.to_csv('customer_clv.csv', index=False)
print("✅ Project 5 complete.")
